# Audit DB Advanced: How to Turn Events into Metrics

**Difficulty: Advanced | ~45 min | Requires Lab 4 (Filtering, Search, and Pagination) done in this Supabase project**

A log of events is useful for *replay* — answering "what happened?". This lab turns it into a
*dashboard* — the aggregate numbers a team actually reads: error rates, runs per hour, cost and
latency per agent, and guardrail failure rates.

The through-line is a single idea that separates the two Postgres features we will use:

> A **VIEW** recomputes its rows every time you read it (compute-on-read).
> A **MATERIALIZED VIEW** stores its rows and stays frozen until you **REFRESH** it.

Step 8 is where they diverge, and it is the whole pay-off of this lab.


In [1]:
!pip install python-dotenv==1.2.3 psycopg2-binary==2.9.12



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Run this first cell to install exactly the versions this lab was built and tested with.

- `python-dotenv` reads our secret `DATABASE_URL` out of the project's `.env` file.
- `psycopg2-binary` is the Postgres driver every audit lab has used so far.

Nothing else is needed — all the heavy lifting (aggregation, percentiles, materialization) happens
inside Postgres, and we only *drive* it from Python.


### Step 1 — Connect to Postgres


In [2]:
import os
from pathlib import Path
from datetime import datetime, timezone

from dotenv import load_dotenv
import psycopg2

env_path = next((p / ".env" for p in [Path.cwd(), *Path.cwd().parents] if (p / ".env").exists()), None)
load_dotenv(env_path)
database_url = os.getenv("DATABASE_URL")
assert database_url, "DATABASE_URL not found - complete README Section 7 setup first."

connection = psycopg2.connect(database_url, connect_timeout=10)
cursor = connection.cursor()
cursor.execute("SELECT version();")
print(cursor.fetchone()[0])


PostgreSQL 17.6 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 15.2.0, 64-bit


Identical to every lab before it: walk *up* from the current folder until `.env` is found, load
`DATABASE_URL`, open one connection, and confirm the server answers by printing its version.

Keep `cursor` and `connection` around — we will reuse them for the whole notebook.


### Step 2 — Confirm the schema is here


In [3]:
# Fail-fast: confirm the Lab 1 + Lab 2 tables exist before we write against them.
cursor.execute("SELECT to_regclass('public.run')")
assert cursor.fetchone()[0] is not None, "run table missing - complete Lab 1 first."
cursor.execute("SELECT to_regclass('public.event')")
assert cursor.fetchone()[0] is not None, "event table missing - complete Lab 1 first."
for table in ["span", "tool_call", "guardrail_event"]:
    cursor.execute("SELECT to_regclass(%s)", (f"public.{table}",))
    assert cursor.fetchone()[0] is not None, f"{table} table missing - complete Lab 2 first."
print("Schema found: run, event, span, tool_call, guardrail_event.")


Schema found: run, event, span, tool_call, guardrail_event.


`to_regclass('public.run')` returns the table's OID if it exists and `None` if it does not. Any
`None` means an earlier lab has not been run, and the assertions stop us before we start writing
against tables that are not there.

Our metrics need the Lab 1 tables (`run`, `event`) for runs and costs, and the Lab 2 child
`guardrail_event` (via `span`) for compliance-style metrics.


### Step 3 — Seed a spread of runs worth measuring


In [4]:
import secrets

# One tag for this entire run-through, so cleanup (Step 9) can find exactly these rows.
lab5_tag = secrets.token_hex(4)

# (agent_name, status, total_cost, started_a_go, duration)
#   agent_name  = a logical agent token (no hyphens) - the sort key our metrics GROUP BY
#   started_a_go = how far in the past the run began (Postgres interval string)
#   duration      = the run's latency; ended_at - started_at equals this interval
seed_runs = [
    ("support",   "success", 0.0210, "2 hours",    "35 seconds"),
    ("support",   "error",   0.0045, "1 hour",     "12 seconds"),
    ("billing",   "success", 0.0900, "30 minutes",  "5 seconds"),
    ("billing",   "timeout", 0.0120, "3 days",     "90 seconds"),
    ("support",   "success", 0.0031, "5 days",     "8 seconds"),
    ("copilot",   "error",   0.1100, "4 hours",    "45 seconds"),
    ("copilot",   "success", 0.0077, "6 days",     "20 seconds"),
    ("support",   "timeout", 0.0012, "12 hours",   "60 seconds"),
    ("billing",   "success", 0.4500, "2 days",     "22 seconds"),
    ("copilot",   "error",   0.0088, "45 minutes", "30 seconds"),
    ("support",   "success", 0.0150, "7 hours",    "15 seconds"),
    ("billing",   "error",   0.0010, "5 hours",    "10 seconds"),
    ("copilot",   "success", 0.0300, "2 hours",    "25 seconds"),
    ("support",   "error",   0.0090, "3 hours",    "40 seconds"),
    ("billing",   "success", 0.0060, "20 minutes",  "6 seconds"),
]

# One guardrail check per run's span: (check_name, outcome) with a deliberate mix.
seed_guards = [
    ("pii_scan",            "pass"),
    ("toxicity_scan",       "fail"),
    ("pii_scan",            "pass"),
    ("hallucination_scan",  "warn"),
    ("toxicity_scan",       "pass"),
    ("pii_scan",            "fail"),
    ("hallucination_scan",  "pass"),
    ("pii_scan",            "pass"),
    ("toxicity_scan",       "fail"),
    ("pii_scan",            "pass"),
    ("hallucination_scan",  "fail"),
    ("toxicity_scan",       "pass"),
    ("pii_scan",            "pass"),
    ("hallucination_scan",  "warn"),
    ("toxicity_scan",       "pass"),
]

inserted = []
for i, (agent, status, cost, ago, duration) in enumerate(seed_runs):
    # agent_name = logical agent + the lab5 tag. split_part('-', 3) recovers the agent.
    name = f"pytest-lab5-{agent}-{lab5_tag}-{i}"
    cursor.execute(
        "INSERT INTO run (agent_name, status, total_cost, started_at, ended_at) "
        "VALUES (%s, %s, %s, now() - (interval %s + interval %s), now() - interval %s) "
        "RETURNING run_id",
        (name, status, cost, ago, duration, ago),
    )
    run_id = cursor.fetchone()[0]
    cursor.execute(
        "INSERT INTO span (run_id, span_name, started_at, ended_at) "
        "VALUES (%s, %s, now() - (interval %s + interval %s), now() - interval %s) "
        "RETURNING span_id",
        (run_id, "handle_request", ago, duration, ago),
    )
    span_id = cursor.fetchone()[0]
    check_name, outcome = seed_guards[i]
    cursor.execute(
        "INSERT INTO guardrail_event (span_id, check_name, outcome, reason) "
        "VALUES (%s, %s, %s, %s)",
        (span_id, check_name, outcome, None),
    )
    inserted.append(run_id)

connection.commit()
print(f"Seeded {len(inserted)} tagged runs under marker pytest-lab5-*-{lab5_tag}-* "
      f"(run ids {inserted[0]}..{inserted[-1]}).")


Seeded 15 tagged runs under marker pytest-lab5-*-2a780e77-* (run ids 1242..1256).


A dashboard is only interesting when there is a *spread* to summarize, so we seed **15 runs** across
three logical agents (`support`, `billing`, `copilot`), mixing `success`, `error`, and `timeout`, and
varying cost and latency per run.

- Every run's `agent_name` embeds both its logical agent and a private tag:
  `pytest-lab5-<agent>-<hex>-<n>` (never another lab's marker). `split_part('-', 3)` recovers the
  agent, which is what Steps 4 and 6 group by.
- The run's `started_at` is pushed into the past by `started_a_go`, and its latency is set by
  `duration` (`ended_at - started_at == duration`).
- Each run's span carries exactly **one** guardrail check, so the pass/fail/warn mix in `seed_guards`
  can be aggregated into a guardrail-failure metric later.

There is no fuzz here: the error rates, costs, and check outcomes are all hand-computable, which is
exactly what we will verify in Step 4.


### Step 4 — Error rate per agent (a plain VIEW)


In [5]:
# A VIEW is compute-on-read: every SELECT re-runs the query inside it.
cursor.execute("DROP VIEW IF EXISTS v_run_error_rate")
cursor.execute("""
    CREATE VIEW v_run_error_rate AS
    SELECT
        split_part(agent_name, '-', 3)                       AS agent_name,
        count(*)                                         AS total_runs,
        count(*) FILTER (WHERE status = 'error')         AS errored_runs,
        round(100.0 * count(*) FILTER (WHERE status = 'error') / count(*), 1) AS error_rate_pct
    FROM run
    WHERE agent_name LIKE 'pytest-lab5-%'
    GROUP BY split_part(agent_name, '-', 3)
    ORDER BY error_rate_pct DESC
""")
connection.commit()

cursor.execute("SELECT agent_name, total_runs, errored_runs, error_rate_pct FROM v_run_error_rate")
for agent, total, errored, pct in cursor.fetchall():
    print(f"  {agent}: {errored}/{total} runs errored = {pct}%")


  copilot: 2/4 runs errored = 50.0%
  support: 2/6 runs errored = 33.3%
  billing: 1/5 runs errored = 20.0%


A **VIEW** is just a named query. Writing `CREATE VIEW` stores the *definition*, not the *answer*;
every time you read it, Postgres re-executes the query.

- `count(*) FILTER (WHERE status = 'error')` counts only errored runs in one pass — no CASE, no subquery.
- `split_part(agent_name, '-', 3)` recovers the logical agent from the tagged name, so `GROUP BY`
  folds the six `support` runs into one row instead of reporting each run separately.
- We scope the view to `pytest-lab5-%` so it reads only *this lab's* synthetic rows. In a production
  dashboard that tag would be dropped and the real agent name grouping would take over.

Do the math by hand and it lines up: `support` has 2 errored of 6 (33.3%), `billing` 1 of 5 (20.0%),
`copilot` 2 of 4 (50.0%). The view returns exactly that.


### Step 5 — Runs per hour: date_trunc


In [6]:
# Bucket each run's start time to its hour, then count per bucket.
cursor.execute("DROP VIEW IF EXISTS v_runs_per_hour")
cursor.execute("""
    CREATE VIEW v_runs_per_hour AS
    SELECT
        date_trunc('hour', started_at) AS hour_bucket,
        count(*)                       AS runs
    FROM run
    WHERE agent_name LIKE 'pytest-lab5-%'
    GROUP BY hour_bucket
    ORDER BY hour_bucket
""")
connection.commit()

cursor.execute("SELECT hour_bucket, runs FROM v_runs_per_hour")
for hour, runs in cursor.fetchall():
    print(f"  {hour.isoformat()}: {runs} run(s)")


  2026-08-22T17:00:00+00:00: 1 run(s)
  2026-08-23T17:00:00+00:00: 1 run(s)
  2026-08-25T17:00:00+00:00: 1 run(s)
  2026-08-26T17:00:00+00:00: 1 run(s)
  2026-08-28T05:00:00+00:00: 1 run(s)
  2026-08-28T10:00:00+00:00: 1 run(s)
  2026-08-28T12:00:00+00:00: 1 run(s)
  2026-08-28T13:00:00+00:00: 1 run(s)
  2026-08-28T14:00:00+00:00: 1 run(s)
  2026-08-28T15:00:00+00:00: 2 run(s)
  2026-08-28T16:00:00+00:00: 2 run(s)
  2026-08-28T17:00:00+00:00: 2 run(s)


`date_trunc('hour', started_at)` snaps every timestamp down to its hour boundary, so runs at
`14:03` and `14:51` fall into the same `14:00` bucket. `GROUP BY hour_bucket` then folds them together.

This is the heart of a time-series dashboard — "what does our hourly throughput look like?" Our
seed spreads runs from `20 minutes` to `6 days` in the past, so the buckets span many distinct hours.


### Step 6 — Cost and latency per agent


In [7]:
# AVG is easy to fool (one slow run inflates it), so add a 95th-percentile tail too.
cursor.execute("DROP VIEW IF EXISTS v_agent_cost_latency")
cursor.execute("""
    CREATE VIEW v_agent_cost_latency AS
    SELECT
        split_part(agent_name, '-', 3)                                       AS agent_name,
        round(avg(total_cost)::numeric, 4)                                   AS avg_cost,
        round(avg(EXTRACT(EPOCH FROM (ended_at - started_at))), 1)           AS avg_latency_s,
        round(percentile_cont(0.95) WITHIN GROUP (ORDER BY total_cost)::numeric, 4) AS p95_cost,
        round(percentile_cont(0.95) WITHIN GROUP (ORDER BY EXTRACT(EPOCH FROM (ended_at - started_at)))::numeric, 1) AS p95_latency_s
    FROM run
    WHERE agent_name LIKE 'pytest-lab5-%'
    GROUP BY split_part(agent_name, '-', 3)
    ORDER BY avg_cost DESC
""")
connection.commit()

cursor.execute("SELECT agent_name, avg_cost, avg_latency_s, p95_cost, p95_latency_s FROM v_agent_cost_latency")
for agent, avg_cost, avg_lat, p95_cost, p95_lat in cursor.fetchall():
    print(f"  {agent}: avg_cost={avg_cost} avg_latency={avg_lat}s p95_cost={p95_cost} p95_latency={p95_lat}s")


  billing: avg_cost=0.1118 avg_latency=26.6s p95_cost=0.3780 p95_latency=76.4s
  copilot: avg_cost=0.0391 avg_latency=30.0s p95_cost=0.0980 p95_latency=42.8s
  support: avg_cost=0.0090 avg_latency=28.3s p95_cost=0.0195 p95_latency=55.0s


Two ideas worth keeping:

- **Latency is a timestamp difference.** `ended_at - started_at` yields an interval, so we
  `EXTRACT(EPOCH FROM ...)` to get plain seconds that are easy to average and render.
- **Averages hide outliers.** One 90-second timeout would drag `billing`'s mean latency up even if
  everything else was fast. `percentile_cont(0.95)` returns the value below which 95% of runs fall —
  the "tail" number a dashboard should show *alongside* the mean.

Both are ordinary ordered-set/aggregate functions — no stored logic required yet. Step 8 is where
storage finally matters.


### Step 7 — Guardrail-fail rate


In [8]:
# Join guardrail checks through spans to their runs, then fail-rate per check.
cursor.execute("DROP VIEW IF EXISTS v_guardrail_fail_rate")
cursor.execute("""
    CREATE VIEW v_guardrail_fail_rate AS
    SELECT
        ge.check_name,
        count(*)                                AS total_checks,
        count(*) FILTER (WHERE ge.outcome = 'fail') AS failed_checks,
        round(100.0 * count(*) FILTER (WHERE ge.outcome = 'fail') / count(*), 1) AS fail_rate_pct
    FROM guardrail_event ge
    JOIN span s ON s.span_id = ge.span_id
    JOIN run r ON r.run_id = s.run_id
    WHERE r.agent_name LIKE 'pytest-lab5-%'
    GROUP BY ge.check_name
    ORDER BY fail_rate_pct DESC
""")
connection.commit()

cursor.execute("SELECT check_name, total_checks, failed_checks, fail_rate_pct FROM v_guardrail_fail_rate")
for check, total, failed, pct in cursor.fetchall():
    print(f"  {check}: {failed}/{total} checks failed = {pct}%")


  toxicity_scan: 2/5 checks failed = 40.0%
  hallucination_scan: 1/4 checks failed = 25.0%
  pii_scan: 1/6 checks failed = 16.7%


`guardrail_event` is a child of `span` (Lab 2), so reaching the *run* means joining through two
tables. The join filters to our tagged runs, then the familiar `FILTER (WHERE outcome = 'fail')`
pattern turns raw check outcomes into a compliance-flavored metric.

Hand-check: `toxicity_scan` failed 2 of its 5 checks (40%), `hallucination_scan` failed 1 of 4 (25%),
`pii_scan` failed 1 of 6 (~16.7%). Every one of these is a plain **VIEW** — cheap to build on our
15 synthetic rows, but increasingly expensive as the real log grows. That cost is the bridge to Step 8.


### Step 8 — The scaling problem, and the MATERIALIZED VIEW answer


In [9]:
# 8a) The whole-log aggregate pays a full scan on every single read.
cursor.execute("EXPLAIN SELECT count(*), count(*) FILTER (WHERE status = 'error') FROM run")
print("8a) EXPLAIN of the whole-log error count:")
for row in cursor.fetchall():
    print("    " + row[0])

# 8b) Snapshot that expensive metric into a MATERIALIZED VIEW (stored, not recomputed).
cursor.execute("DROP MATERIALIZED VIEW IF EXISTS mv_global_error_rate")
cursor.execute("""
    CREATE MATERIALIZED VIEW mv_global_error_rate AS
    SELECT
        count(*)                                AS total_runs,
        count(*) FILTER (WHERE status = 'error') AS errored_runs,
        round(100.0 * count(*) FILTER (WHERE status = 'error') / count(*), 2) AS error_rate_pct
    FROM run
""")
connection.commit()

# 8c) Reading it is instant - it is a stored snapshot, not a recomputed query.
cursor.execute("SELECT total_runs, errored_runs, error_rate_pct FROM mv_global_error_rate")
before = cursor.fetchone()
print(f"8c) Matview (initial snapshot): total={before[0]} errored={before[1]} pct={before[2]}")

# 8d) A new run arrives - the matview is STALE until we refresh it.
cursor.execute(
    "INSERT INTO run (agent_name, status, total_cost) VALUES (%s, %s, %s)",
    (f"pytest-lab5-{lab5_tag}-NEW", "error", 0.02),
)
connection.commit()
cursor.execute("SELECT total_runs, errored_runs, error_rate_pct FROM mv_global_error_rate")
stale = cursor.fetchone()
print(f"8d) Matview after inserting a run, BEFORE refresh: total={stale[0]} errored={stale[1]} pct={stale[2]}")
assert before == stale, "matview should be stale (unchanged) until refreshed"

# 8e) REFRESH makes it current.
cursor.execute("REFRESH MATERIALIZED VIEW mv_global_error_rate")
connection.commit()
cursor.execute("SELECT total_runs, errored_runs, error_rate_pct FROM mv_global_error_rate")
after = cursor.fetchone()
print(f"8e) Matview after REFRESH: total={after[0]} errored={after[1]} pct={after[2]}")
print("    Stale before refresh; current after. That is the materialized-view trade-off.")


8a) EXPLAIN of the whole-log error count:
    Aggregate  (cost=1.12..1.13 rows=1 width=16)
      ->  Seq Scan on run  (cost=0.00..1.07 rows=7 width=10)


8c) Matview (initial snapshot): total=22 errored=5 pct=22.73


8d) Matview after inserting a run, BEFORE refresh: total=22 errored=5 pct=22.73


8e) Matview after REFRESH: total=23 errored=6 pct=26.09
    Stale before refresh; current after. That is the materialized-view trade-off.


This is the intellectual peak, so read closely.

1. **8a** `EXPLAIN` shows the problem. Counting errors over the whole `run` table is a **Seq Scan**
   that touches every row. With a plain **VIEW**, that cost is paid on *every dashboard refresh*,
   even though the answer barely changes between reads.
2. **8b** A **MATERIALIZED VIEW** stores the *answer* on disk. Building it runs the query once; reading
   it afterwards is just reading a table — no expensive re-scan.
3. **8c–8d** The trade-off: the stored snapshot goes **stale** the moment new rows arrive. After we
   insert one more run, the matview still reports the old totals — it does *not* magically know about
   the new row. The `assert before == stale` proves it stayed frozen.
4. **8e** `REFRESH MATERIALIZED VIEW` re-runs the query and swaps in new data, so it becomes current.

So: **views** = always correct, recompute each read; **materialized views** = instant to read, but you
must choose *when* to refresh. Dashboards that tolerate a few minutes of staleness (like this one)
happily trade correctness-lag for speed.


### Step 9 — Cleanup and recap


In [10]:
# Drop every object this lab created (views + the materialized view).
for stmt in [
    "DROP VIEW IF EXISTS v_guardrail_fail_rate",
    "DROP VIEW IF EXISTS v_agent_cost_latency",
    "DROP VIEW IF EXISTS v_runs_per_hour",
    "DROP VIEW IF EXISTS v_run_error_rate",
    "DROP MATERIALIZED VIEW IF EXISTS mv_global_error_rate",
]:
    cursor.execute(stmt)
connection.commit()

# Delete only this lab's tagged rows - children first (FK order), then parents.
cursor.execute(
    "DELETE FROM guardrail_event WHERE span_id IN "
    "(SELECT span_id FROM span WHERE run_id IN (SELECT run_id FROM run WHERE agent_name LIKE %s))",
    ("pytest-lab5-%",),
)
cursor.execute(
    "DELETE FROM tool_call WHERE span_id IN "
    "(SELECT span_id FROM span WHERE run_id IN (SELECT run_id FROM run WHERE agent_name LIKE %s))",
    ("pytest-lab5-%",),
)
cursor.execute(
    "DELETE FROM span WHERE run_id IN (SELECT run_id FROM run WHERE agent_name LIKE %s)",
    ("pytest-lab5-%",),
)
cursor.execute(
    "DELETE FROM event WHERE run_id IN (SELECT run_id FROM run WHERE agent_name LIKE %s)",
    ("pytest-lab5-%",),
)
cursor.execute("DELETE FROM run WHERE agent_name LIKE %s", ("pytest-lab5-%",))
connection.commit()

cursor.execute("SELECT count(*) FROM run WHERE agent_name LIKE %s", ("pytest-lab5-%",))
leftover = cursor.fetchone()[0]

cursor.close()
connection.close()

print(f"Cleanup done - {leftover} tagged runs left behind; views + materialized view dropped.")
print("Lab 5 complete: named metrics from a view vs a materialized view.")


Cleanup done - 0 tagged runs left behind; views + materialized view dropped.
Lab 5 complete: named metrics from a view vs a materialized view.


We leave the schema exactly as Lab 1 + Lab 2 shipped it:

- **Drop** the four views and the materialized view we created, in no particular dependency order
  (none reference another). `IF EXISTS` makes it safe to re-run.
- **Delete** only our `pytest-lab5-%` rows, children before parents so foreign keys are satisfied.
- **Verify** nothing tagged remains, then close the connection.

**What this lab proved:** turning raw events into metrics is a matter of named aggregate queries
(views), bucketing time (`date_trunc`), resisting averages with percentiles, and — when the log
grows — snapshotting expensive answers into materialized views that you refresh on your own
schedule. The next lab, *Lab 6*, builds on this measurement tooling with projections and forecasting.

Seen through the harness lens, these named views are what a dashboard reads to show cost, error rate, and latency per agent without rescanning the raw log on every refresh.
